## API Call

In [1]:
import requests
import pandas as pd


def get_nsf_awards(full_name, rpp=25):

    full_name = str(full_name).strip()

    print(f"Searching NSF: {full_name}")

    def search_awards(search_parameter):

        awards_all = []
        offset = 0

        while True:

            params = {
                search_parameter: full_name,
                "rpp": rpp,
                "offset": offset
            }

            response = requests.get(
                "https://api.nsf.gov/services/v1/awards.json",
                params=params,
                timeout=30
            )

            response.raise_for_status()

            awards = response.json()["response"].get("award", [])

            if not awards:
                break

            awards_all.extend(awards)

            if len(awards) < rpp:
                break

            offset += rpp

        return awards_all

    # Search PI and Co-PI awards
    pi_awards = search_awards("pdPIName")
    copi_awards = search_awards("coPDPI")

    # Add role
    for award in pi_awards:
        award["Role"] = "PI"

    for award in copi_awards:
        award["Role"] = "Co-PI"

    # Combine
    df = pd.DataFrame(pi_awards + copi_awards)

    # Add searched person's information
    df["Full Name"] = full_name

    # Desired output columns
    output_columns = [
        "Full Name",
        "Participants",
        "Role",
        "Title",
        "Abstract",
        "Award Type",
        "Journal",
        "Amount",
        "NSF Email",
        "Institute",
        "Start Date",
        "End Date"
    ]

    # NSF field names
    columns = {
        "title": "Title",
        "abstractText": "Abstract",
        "transType": "Award Type",
        "jrnl": "Journal",
        "estimatedTotalAmt": "Amount",
        "piEmail": "NSF Email",
        "awardee": "Institute",
        "startDate": "Start Date",
        "expDate": "End Date"
    }

    df = df.rename(columns=columns)

    # Ensure every desired column exists
    for col in output_columns:
        if col not in df.columns:
            df[col] = pd.NA

    return df[output_columns]

In [2]:
def make_full_name(first_name, last_name):

    name_parts = [first_name, last_name]

    name_parts = [
        str(x).strip()
        for x in name_parts
        if pd.notna(x)
        and str(x).strip() != ""
        and str(x).strip().lower() != "nan"
    ]

    return " ".join(name_parts)

In [3]:
df = pd.read_csv("mhti_applicants.csv")[["first_name", "last_name"]]

df["name"] = df.apply(
    lambda row: make_full_name(
        row["first_name"],
        row["last_name"]
    ),
    axis=1
)

print(df)

    first_name   last_name              name
0        Elena      Agapie      Elena Agapie
1         Aima    Ahonkhai     Aima Ahonkhai
2     Nicholas       Allan    Nicholas Allan
3     Nicholas       Allen    Nicholas Allen
4        Nabil   Alshurafa   Nabil Alshurafa
..         ...         ...               ...
535      mehdi  boukhechba  mehdi boukhechba
536      Rocio  de la Vega  Rocio de la Vega
537      sarah        gray        sarah gray
538       shan         lin          shan lin
539      James    suliburk    James suliburk

[540 rows x 3 columns]


In [4]:
import time

def get_nsf_awards_for_people(people, delay=0.2):

    all_results = []

    for _, person in people.iterrows():

        full_name = person["name"]

        try:

            result = get_nsf_awards(
                full_name=full_name
            )

            if not result.empty:
                all_results.append(result)

                print(
                    f"  Found {len(result)} matching award(s)"
                )

            else:
                print("  No matching awards found")

        except requests.exceptions.RequestException as e:

            print(
                f"  API error for "
                f"{full_name}: {e}"
            )

        except Exception as e:

            print(
                f"  Error for "
                f"{full_name}: {e}"
            )

        time.sleep(delay)

    if not all_results:
        return pd.DataFrame(
            columns=[
                "Title",
                "Abstract",
                "Award Type",
                "Amount",
                "Email",
                "Institute",
                "Date"
            ]
        )

    return pd.concat(
        all_results,
        ignore_index=True
    )

In [5]:
results = get_nsf_awards_for_people(df)

Searching NSF: Elena Agapie
  Found 2 matching award(s)
Searching NSF: Aima Ahonkhai
  No matching awards found
Searching NSF: Nicholas Allan
  No matching awards found
Searching NSF: Nicholas Allen
  Found 2 matching award(s)
Searching NSF: Nabil Alshurafa
  Found 5 matching award(s)
Searching NSF: Ashwin Ashok
  Found 7 matching award(s)
Searching NSF: Arjun Athreya
  Found 1 matching award(s)
Searching NSF: Sherif Badawy
  No matching awards found
Searching NSF: Kara Bagot
  No matching awards found
Searching NSF: Jeannie Bailey
  No matching awards found
Searching NSF: Ryan Bailey
  Found 8 matching award(s)
Searching NSF: Yael Bar-Zeev
  No matching awards found
Searching NSF: John Batsis
  No matching awards found
Searching NSF: Laura Bellows
  No matching awards found
Searching NSF: Natalie Benda
  No matching awards found
Searching NSF: Jennifer Blain Christen
  No matching awards found
Searching NSF: Luisa Blanco
  Found 1 matching award(s)
Searching NSF: Nick Bott
  No matchi

In [6]:
results["Role"].unique()

array(['PI', 'Co-PI'], dtype=object)

In [7]:
results

,Full Name,Participants,Role,Title,Abstract,Award Type,Journal,Amount,NSF Email,Institute,Start Date,End Date
0,Elena Agapie,NaN,PI,Travel: NSF Student Travel Grant for the 2025 ...,This award provides funding to support about n...,Standard Grant,NaN,23788,eagapie@uci.edu,UNIVERSITY OF CALIFORNIA IRVINE,06/15/2025,05/31/2026
1,Elena Agapie,NaN,PI,EAGER: Curating and representing mental health...,This research addresses the growing need to su...,Standard Grant,[{'artTitl': 'Conducting Research at the Inter...,149989,eagapie@uci.edu,UNIVERSITY OF CALIFORNIA IRVINE,10/01/2022,12/31/2025
2,Nicholas Allen,NaN,Co-PI,Modeling Developmental Change: Practical Integ...,"Neuroimaging techniques, including structural ...",Standard Grant,NaN,19900,mbyrne@uoregon.edu,UNIVERSITY OF OREGON,06/01/2017,05/31/2018
3,Nicholas Allen,NaN,Co-PI,The Evolution of Cognitive Capactities for Coo...,This research project studies the evolution of...,Standard Grant,NaN,126164,jorbell@uoregon.edu,UNIVERSITY OF OREGON,08/01/1998,07/31/2001
4,Nabil Alshurafa,NaN,PI,Travel: NSF Student Travel Grant for the 2024 ...,This project is to support student travel at t...,Standard Grant,NaN,20000,nabil@northwestern.edu,NORTHWESTERN UNIVERSITY,07/15/2024,06/30/2025
...,...,...,...,...,...,...,...,...,...,...,...,...
1054,shan lin,NaN,PI,CPS: Synergy: Collaborative Research: Multiple...,Cyber physical systems (CPSs) are merging into...,Standard Grant,"[{'artPageNum': '100', 'artTitl': 'Reducing Wa...",199996,shan.x.lin@stonybrook.edu,TEMPLE UNIVERSITY-OF THE COMMONWEALTH SYSTEM O...,10/01/2012,06/30/2015
1055,shan lin,NaN,PI,SHB: Type I (EXP): Collaborative Research: Het...,Telemedicine technologies offer the opportunit...,Standard Grant,NaN,299554,shan.x.lin@stonybrook.edu,TEMPLE UNIVERSITY-OF THE COMMONWEALTH SYSTEM O...,09/01/2012,09/30/2014
1056,shan lin,NaN,PI,NeTS: Small: Collaborative Research: Non-Isotr...,The objective of this proposal is to thoroughl...,Standard Grant,"[{'artPageNum': '0', 'artTitl': 'An Automatic,...",270655,shan.x.lin@stonybrook.edu,TEMPLE UNIVERSITY-OF THE COMMONWEALTH SYSTEM O...,07/01/2012,10/31/2014
1057,shan lin,NaN,Co-PI,NSF CAREER Workshop,This effort will organize an NSF CAREER Propos...,Standard Grant,NaN,49999,jiewu@temple.edu,TEMPLE UNIVERSITY-OF THE COMMONWEALTH SYSTEM O...,12/01/2012,11/30/2015


In [8]:
results.to_excel("NSF_raw3.xlsx", index=False)

## Merge with Demographic Data

In [9]:
import pandas as pd

df_nsf = pd.read_excel("NSF_raw3.xlsx")
df_demo = pd.read_csv("mhti_applicants.csv")

df_demo["Full Name"] = df_demo.apply(
    lambda row: make_full_name(
        row["first_name"],
        row["last_name"]
    ),
    axis=1
)

df_demo = df_demo.drop(columns=['first_name', 'last_name'])

In [10]:
df_nsf2 = df_nsf.copy()
df_demo2 = df_demo.copy()

# --- Merge onto NSF data ---

df_nsf2 = df_nsf2.merge(
    df_demo,
    on=["Full Name"],
    how="left"
)

df_nsf2

,Full Name,Participants,Role,Title,Abstract,Award Type,Journal,Amount,NSF Email,Institute,...,cohorts_applied,statuses,n_applications,best_committee_score,any_alternate_withdrawn,scored_application,tier,author_id,works_count,openalex_linked_confidently
0,Elena Agapie,NaN,PI,Travel: NSF Student Travel Grant for the 2025 ...,This award provides funding to support about n...,Standard Grant,NaN,23788,eagapie@uci.edu,UNIVERSITY OF CALIFORNIA IRVINE,...,2021,Recipient,1,3.6,0,1,A_orcid,A5015415555,40.0,1
1,Elena Agapie,NaN,PI,EAGER: Curating and representing mental health...,This research addresses the growing need to su...,Standard Grant,[{'artTitl': 'Conducting Research at the Inter...,149989,eagapie@uci.edu,UNIVERSITY OF CALIFORNIA IRVINE,...,2021,Recipient,1,3.6,0,1,A_orcid,A5015415555,40.0,1
2,Nicholas Allen,NaN,Co-PI,Modeling Developmental Change: Practical Integ...,"Neuroimaging techniques, including structural ...",Standard Grant,NaN,19900,mbyrne@uoregon.edu,UNIVERSITY OF OREGON,...,2017,Awarded,1,1.0,0,1,D_ambiguous,A5151652082,1.0,0
3,Nicholas Allen,NaN,Co-PI,The Evolution of Cognitive Capactities for Coo...,This research project studies the evolution of...,Standard Grant,NaN,126164,jorbell@uoregon.edu,UNIVERSITY OF OREGON,...,2017,Awarded,1,1.0,0,1,D_ambiguous,A5151652082,1.0,0
4,Nabil Alshurafa,NaN,PI,Travel: NSF Student Travel Grant for the 2024 ...,This project is to support student travel at t...,Standard Grant,NaN,20000,nabil@northwestern.edu,NORTHWESTERN UNIVERSITY,...,2018,Awarded,1,1.0,0,1,A_orcid,A5001385856,129.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1110,shan lin,NaN,PI,CPS: Synergy: Collaborative Research: Multiple...,Cyber physical systems (CPSs) are merging into...,Standard Grant,"[{'artPageNum': '100', 'artTitl': 'Reducing Wa...",199996,shan.x.lin@stonybrook.edu,TEMPLE UNIVERSITY-OF THE COMMONWEALTH SYSTEM O...,...,2022,Declined,1,1.0,0,1,D_ambiguous,A5017265376,103.0,0
1111,shan lin,NaN,PI,SHB: Type I (EXP): Collaborative Research: Het...,Telemedicine technologies offer the opportunit...,Standard Grant,NaN,299554,shan.x.lin@stonybrook.edu,TEMPLE UNIVERSITY-OF THE COMMONWEALTH SYSTEM O...,...,2022,Declined,1,1.0,0,1,D_ambiguous,A5017265376,103.0,0
1112,shan lin,NaN,PI,NeTS: Small: Collaborative Research: Non-Isotr...,The objective of this proposal is to thoroughl...,Standard Grant,"[{'artPageNum': '0', 'artTitl': 'An Automatic,...",270655,shan.x.lin@stonybrook.edu,TEMPLE UNIVERSITY-OF THE COMMONWEALTH SYSTEM O...,...,2022,Declined,1,1.0,0,1,D_ambiguous,A5017265376,103.0,0
1113,shan lin,NaN,Co-PI,NSF CAREER Workshop,This effort will organize an NSF CAREER Propos...,Standard Grant,NaN,49999,jiewu@temple.edu,TEMPLE UNIVERSITY-OF THE COMMONWEALTH SYSTEM O...,...,2022,Declined,1,1.0,0,1,D_ambiguous,A5017265376,103.0,0


In [11]:
df_nsf2.to_excel("NSF_demographic3.xlsx", index=False)